# Preparación de datos — Trufi App Cochabamba
## CRISP-DM Fase 3 · Tabla minable por celda H3 (encuadre predictivo)

> **Cambio de área respecto a la Etapa 1.** El área de estudio **ya no** es la envolvente convexa del feed GTFS (como se documentó en D-009 de la Etapa 1). Se redefine como la **envolvente convexa de los orígenes de consultas válidas + buffer de 1 km**. Razón: el objetivo es predictivo y el área debe corresponder al **soporte espacial de los datos**, no a un polígono operativo externo. GTFS deja de definir el área y pasa a ser una variable de contraste reservada para la evaluación. La decisión previa (D-009) queda marcada como **superada por D-018** en `DECISIONES.md`.

**Pregunta:** ¿Cómo estimar el número esperado de consultas de ruta de Trufi App por celda H3 a partir de la población, la ubicación y el contexto territorial de cada celda, mediante técnicas geoespaciales de ciencia de datos?

| Tarea | Contenido | Salida |
|---|---|---|
| T1 | Área = envolvente de orígenes válidos + 1 km | `area_estudio.geojson` |
| T2 | Diagnóstico del filtro de 30 km | `filtro_30km_diagnostico.csv`, `sensibilidad_umbral_km.csv` |
| T3 | Limpieza a nivel consulta | `queries_limpias.parquet`, `tabla_flujo_limpieza.csv` |
| T4 | Tabla por celda (con celdas en cero) | — (se escribe una vez al final de T5) |
| T5 | Variables territoriales y de contraste | `tabla_minable.parquet`, `tabla_minable_dtypes.json` |
| T6 | Reserva de prueba por bloques H3 res 6 | `test_blocks.csv` |
| T7 | Auditoría de *leakage* L1–L8 | `auditoria_leakage.csv` |
| T8 | Verificaciones, resúmenes y figuras | `resumen_*.csv`, `diccionario_datos.csv`, `figuras/*.png` |
| T9 | README de la fase | `README.md` |

Vocabulario: **celda H3** (res 8), **bloque H3 res 6**, **área de estudio**, **consulta válida**, `gtfs_covered`, `dist_stop_m`. Nombres de columnas según D-019.

In [1]:
import sys
from pathlib import Path


def _encontrar_raiz(inicio: Path) -> Path:
    for carpeta in [inicio, *inicio.parents]:
        if (carpeta / "pyproject.toml").exists():
            return carpeta
    raise RuntimeError("No se encontró pyproject.toml")


RAIZ = _encontrar_raiz(Path.cwd())
SRC_DIR = RAIZ / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

In [2]:
from trufi_ds.notebook_setup import *  # noqa: F403  pl, pd, np, plt, h3, crear_directorios, guardar_figura, PROJECT_ROOT
from trufi_ds.config import (
    QUERIES_PARQUET, GTFS_DIR, KONTUR_2023_GPKG_GZ, KONTUR_2022_GPKG_GZ, PLAZA_14_SEPTIEMBRE, UTM_EPSG,
)
from trufi_ds import preparation as prep

import json
import geopandas as gpd
import libpysal
from esda.moran import Moran
from scipy.stats import spearmanr

SEMILLA = 42
DIST_MAX_M = 30_000            # D-015
BUFFER_AREA_M = 1_000          # D-018
POP_MIN = 10                   # D-108
UMBRAL_COBERTURA_M = 500       # D-014
FRACCION_PRUEBA = 0.20         # D-106

np.random.seed(SEMILLA)
SUB = "02_preparacion"
crear_directorios(SUB, f"{SUB}/figuras")
REPORTE = PROJECT_ROOT / "reports" / SUB
INTERIM = PROJECT_ROOT / "data" / "interim"
PROCESSED = PROJECT_ROOT / "data" / "processed"
plt.rcParams.update({"figure.dpi": 100, "axes.spines.top": False, "axes.spines.right": False})

### Carga y nomenclatura (D-019)

Se leen las consultas consolidadas de la Etapa 1 y se renombran las coordenadas.
Se agregan las celdas H3 de origen (una vez por par de coordenadas distinto) y los
flags de coordenadas inválidas y de usuario anómalo (D-008).

In [3]:
consultas = pl.read_parquet(QUERIES_PARQUET).rename(prep.RENAME_QUERIES)
N_CRUDAS = consultas.height

anomalos = (pl.read_csv(PROJECT_ROOT / "reports/01_data_understanding/usuarios_anomalos.csv")
            .filter(pl.col("flag_anomalo"))["userID"])

lat, lon = pl.col("lat_origin"), pl.col("lon_origin")
B = prep.BBOX_BOLIVIA
consultas = consultas.with_columns(
    ((lat == 0.0) | (lon == 0.0)).alias("flag_zero"),
    (~(lat.is_between(B["lat_min"], B["lat_max"]) & lon.is_between(B["lon_min"], B["lon_max"]))).alias("flag_bolivia_bbox_out"),
    pl.col("userID").is_in(anomalos.implode()).alias("flag_anomalous_user"),
    (pl.col("distancia") > DIST_MAX_M).alias("flag_long_trip"),
)
consultas = consultas.with_columns(
    (pl.col("flag_bolivia_bbox_out") & ~pl.col("flag_zero")).alias("flag_outside_bolivia")
).drop("flag_bolivia_bbox_out")

coords_ok = ~pl.col("flag_zero") & ~pl.col("flag_outside_bolivia")
consultas = pl.concat([
    prep.add_cells(consultas.filter(coords_ok), "lat_origin", "lon_origin", "h3_origin"),
    consultas.filter(~coords_ok).with_columns(pl.lit(None, dtype=pl.Utf8).alias("h3_origin")),
], how="vertical_relaxed")
print(f"Consultas crudas: {N_CRUDAS:,}  |  usuarios anómalos (D-008): {anomalos.len()}")

Consultas crudas: 1,927,675  |  usuarios anómalos (D-008): 2


## T1 · Área de estudio = envolvente de orígenes válidos + 1 km (D-018)

**Consulta válida para definir el área:** coordenadas de origen correctas, `distancia ≤ 30 km` y usuario no anómalo.

Una envolvente convexa es sensible a puntos aislados: 11 orígenes válidos caen en La Paz, Oruro y Santa Cruz
(viajes cortos hechos en otras ciudades), y con ellos la envolvente abarcaría ~41.000 km² de Bolivia. Para que el
área siga dependiendo **solo de ubicaciones** (no de conteos), se toma la **componente espacial principal**: el
mayor conjunto de celdas H3 r8 ocupadas por al menos un origen válido y conectadas por contigüidad H3
(`grid_disk(c, 1)`, la misma vecindad que se usa en todo el análisis). La envolvente se calcula sobre los
orígenes de esas celdas. La cantidad de consultas por celda no interviene.

In [4]:
validas_area = consultas.filter(
    coords_ok & ~pl.col("flag_long_trip") & ~pl.col("flag_anomalous_user")
)
celdas_ocupadas = set(validas_area["h3_origin"].unique().to_list())

variantes = []
area_por_k = {}
for k in [None, 1, 2, 3]:
    celdas_k = celdas_ocupadas if k is None else prep.main_component(celdas_ocupadas, k)
    pts = validas_area.filter(pl.col("h3_origin").is_in(list(celdas_k))).select("lat_origin", "lon_origin").unique()
    geom, km2 = prep.hull_area(pts["lat_origin"].to_numpy(), pts["lon_origin"].to_numpy(), BUFFER_AREA_M)
    area_por_k[k] = geom
    dentro = prep.points_in(geom, validas_area["lat_origin"].to_numpy(), validas_area["lon_origin"].to_numpy())
    variantes.append({
        "variante": "todas_las_celdas" if k is None else f"componente_principal_k{k}",
        "celdas_ocupadas": len(celdas_k), "area_km2": round(km2, 1),
        "consultas_validas_dentro": int(dentro.sum()),
        "pct_consultas_validas_dentro": round(dentro.mean() * 100, 3),
        "adoptada": k == 1,
    })
tabla_variantes = pl.DataFrame(variantes)
tabla_variantes.write_csv(REPORTE / "area_estudio_variantes.csv")
tabla_variantes

variante,celdas_ocupadas,area_km2,consultas_validas_dentro,pct_consultas_validas_dentro,adoptada
str,i64,f64,i64,f64,bool
"""todas_las_celdas""",1146,40967.3,1920618,100.0,false
"""componente_principal_k1""",835,1181.4,1918899,99.91,true
"""componente_principal_k2""",1020,1903.2,1920156,99.976,false
"""componente_principal_k3""",1079,2304.6,1920516,99.995,false


In [5]:
AREA_ESTUDIO = area_por_k[1]
gpd.GeoDataFrame({"nombre": ["area_estudio_etapa2"], "regla": ["hull orígenes válidos (componente k=1) + 1 km"]},
                 geometry=[AREA_ESTUDIO], crs="EPSG:4326").to_file(REPORTE / "area_estudio.geojson", driver="GeoJSON")

AREA_GTFS = gpd.read_file(PROJECT_ROOT / "reports/01_data_understanding/area_estudio_gtfs.geojson").geometry.iloc[0]
km2_etapa1 = prep.area_km2(AREA_GTFS)
km2_etapa2 = prep.area_km2(AREA_ESTUDIO)
km2_interseccion = prep.area_km2(AREA_GTFS.intersection(AREA_ESTUDIO))

en_area = prep.points_in(AREA_ESTUDIO, consultas["lat_origin"].to_numpy(), consultas["lon_origin"].to_numpy())
consultas = consultas.with_columns((pl.lit(pl.Series(en_area)) & coords_ok).alias("in_area"))
n_coords_ok = consultas.filter(coords_ok).height
n_en_area = int(consultas["in_area"].sum())

comparacion_area = pl.DataFrame([
    {"medida": "area_etapa1_hull_gtfs_km2", "valor": round(km2_etapa1, 1)},
    {"medida": "area_etapa2_hull_origenes_km2", "valor": round(km2_etapa2, 1)},
    {"medida": "diferencia_km2", "valor": round(km2_etapa2 - km2_etapa1, 1)},
    {"medida": "interseccion_km2", "valor": round(km2_interseccion, 1)},
    {"medida": "pct_etapa1_conservado", "valor": round(km2_interseccion / km2_etapa1 * 100, 1)},
    {"medida": "consultas_coord_ok", "valor": n_coords_ok},
    {"medida": "consultas_coord_ok_dentro_area_etapa2", "valor": n_en_area},
    {"medida": "pct_consultas_dentro_area_etapa2", "valor": round(n_en_area / n_coords_ok * 100, 2)},
])
comparacion_area.write_csv(REPORTE / "area_comparacion_etapas.csv")
comparacion_area

medida,valor
str,f64
"""area_etapa1_hull_gtfs_km2""",1372.9
"""area_etapa2_hull_origenes_km2""",1181.4
"""diferencia_km2""",-191.4
"""interseccion_km2""",1095.9
"""pct_etapa1_conservado""",79.8
"""consultas_coord_ok""",1.927663e6
"""consultas_coord_ok_dentro_area…",1.924094e6
"""pct_consultas_dentro_area_etap…",99.81


## T2 · Diagnóstico previo al filtro de 30 km (D-015)

Se diagnostica sobre las consultas con coordenadas correctas y sin copias duplicadas, antes de aplicar el área y
el umbral. Se reporta cuántas superan 30 km, desde qué municipio se originan y qué parte tiene el destino fuera
del área de estudio (viaje interurbano).

In [6]:
cols_exacto = [c for c in consultas.columns
               if c not in ("source_file", "source_batch", "source_encoding") and not c.startswith("flag_")
               and c not in ("h3_origin", "in_area")]
consultas = consultas.with_columns(
    (~pl.struct(cols_exacto).is_first_distinct()).alias("flag_duplicate_copy")
)
base_t2 = consultas.filter(coords_ok & ~pl.col("flag_duplicate_copy"))
dest_en_area = prep.points_in(AREA_ESTUDIO, base_t2["lat_destination"].to_numpy(), base_t2["lon_destination"].to_numpy())
base_t2 = base_t2.with_columns(pl.Series("dest_in_area", dest_en_area),
                               pl.col("origin_municipio").map_elements(prep.fix_mojibake, return_dtype=pl.Utf8))
largas = base_t2.filter(pl.col("flag_long_trip"))

diag = [
    {"dimension": "total", "categoria": "consultas_coord_ok_sin_duplicados", "n": base_t2.height, "pct": 100.0},
    {"dimension": "total", "categoria": "distancia_mayor_30km", "n": largas.height,
     "pct": round(largas.height / base_t2.height * 100, 3)},
    {"dimension": "largas", "categoria": "origen_dentro_area", "n": int(largas["in_area"].sum()),
     "pct": round(largas["in_area"].mean() * 100, 2)},
    {"dimension": "largas", "categoria": "destino_fuera_area", "n": int((~largas["dest_in_area"]).sum()),
     "pct": round((~largas["dest_in_area"]).mean() * 100, 2)},
    {"dimension": "largas", "categoria": "origen_y_destino_dentro_area",
     "n": int((largas["in_area"] & largas["dest_in_area"]).sum()),
     "pct": round((largas["in_area"] & largas["dest_in_area"]).mean() * 100, 2)},
]
for fila in largas.group_by("origin_municipio").len().sort("len", descending=True).head(12).iter_rows(named=True):
    diag.append({"dimension": "largas_por_municipio_origen", "categoria": fila["origin_municipio"],
                 "n": fila["len"], "pct": round(fila["len"] / largas.height * 100, 2)})
filtro_diag = pl.DataFrame(diag)
filtro_diag.write_csv(REPORTE / "filtro_30km_diagnostico.csv")
filtro_diag

dimension,categoria,n,pct
str,str,i64,f64
"""total""","""consultas_coord_ok_sin_duplica…",1927603,100.0
"""total""","""distancia_mayor_30km""",6812,0.353
"""largas""","""origen_dentro_area""",4962,72.84
"""largas""","""destino_fuera_area""",4738,69.55
"""largas""","""origen_y_destino_dentro_area""",254,3.73
…,…,…,…
"""largas_por_municipio_origen""","""Tiquipaya""",151,2.22
"""largas_por_municipio_origen""","""Villa José Quintín Mendoza""",126,1.85
"""largas_por_municipio_origen""","""Capinota""",89,1.31


**Sensibilidad del umbral (20/30/50 km).** Con el área fija, se recalcula cuántas consultas quedan y cuánto cambia
el orden de las celdas (Spearman de `query_count` frente al umbral adoptado de 30 km).

In [7]:
base_sens = consultas.filter(pl.col("in_area") & ~pl.col("flag_duplicate_copy") & ~pl.col("flag_anomalous_user"))
conteo_30 = base_sens.filter(pl.col("distancia") <= 30_000).group_by("h3_origin").len()
sens = []
for umbral_km in [20, 30, 50]:
    sub = base_sens.filter(pl.col("distancia") <= umbral_km * 1000)
    conteo = sub.group_by("h3_origin").len()
    comp = conteo_30.join(conteo, on="h3_origin", how="full", coalesce=True).fill_null(0)
    sens.append({
        "umbral_km": umbral_km, "consultas_excluidas": base_sens.height - sub.height,
        "pct_excluidas": round((base_sens.height - sub.height) / base_sens.height * 100, 3),
        "consultas_finales": sub.height, "celdas_con_consultas": conteo.height,
        "spearman_vs_30km": round(spearmanr(comp["len"], comp["len_right"]).statistic, 5),
    })
sens_umbral = pl.DataFrame(sens)
sens_umbral.write_csv(REPORTE / "sensibilidad_umbral_km.csv")
sens_umbral

umbral_km,consultas_excluidas,pct_excluidas,consultas_finales,celdas_con_consultas,spearman_vs_30km
i64,i64,f64,i64,i64,f64
20,15816,0.822,1907986,853,0.98966
30,4962,0.258,1918840,885,1.0
50,258,0.013,1923544,889,0.99981


## T3 · Limpieza a nivel consulta

Orden fijo: (0,0) → fuera de Bolivia → copias de duplicados exactos (D-013) → origen fuera del área (D-018) →
`distancia > 30 km` (D-015) → usuarios anómalos (D-008). No se imputa nada.

In [8]:
pasos = [
    ("1", "origen (0,0)", "flag_zero", "D-005"),
    ("2", "origen fuera de Bolivia", "flag_outside_bolivia", "D-005"),
    ("3", "copia de duplicado exacto", "flag_duplicate_copy", "D-013"),
    ("4", "origen fuera del área de estudio", "fuera_area", "D-018"),
    ("5", "distancia > 30 km", "flag_long_trip", "D-015"),
    ("6", "usuario anómalo", "flag_anomalous_user", "D-008"),
]
actual = consultas.with_columns((~pl.col("in_area")).alias("fuera_area"))
flujo = [{"paso": "0", "regla": "consultas crudas", "filas_antes": N_CRUDAS, "filas_eliminadas": 0,
          "filas_despues": N_CRUDAS, "pct_eliminadas_del_total": 0.0, "decision": "—"}]
for paso, regla, col, decision in pasos:
    antes = actual.height
    actual = actual.filter(~pl.col(col))
    flujo.append({"paso": paso, "regla": regla, "filas_antes": antes, "filas_eliminadas": antes - actual.height,
                  "filas_despues": actual.height,
                  "pct_eliminadas_del_total": round((antes - actual.height) / N_CRUDAS * 100, 4), "decision": decision})
tabla_flujo = pl.DataFrame(flujo)
tabla_flujo.write_csv(REPORTE / "tabla_flujo_limpieza.csv")

consultas_limpias = actual.drop([c for c in actual.columns if c.startswith("flag_")] + ["fuera_area", "in_area"])
consultas_limpias.write_parquet(INTERIM / "queries_limpias.parquet", compression="snappy")
N_LIMPIAS = consultas_limpias.height
print(f"Consultas válidas (limpias): {N_LIMPIAS:,} ({N_LIMPIAS / N_CRUDAS * 100:.2f} % de las crudas)")
tabla_flujo

Consultas válidas (limpias): 1,918,840 (99.54 % de las crudas)


paso,regla,filas_antes,filas_eliminadas,filas_despues,pct_eliminadas_del_total,decision
str,str,i64,i64,i64,f64,str
"""0""","""consultas crudas""",1927675,0,1927675,0.0,"""—"""
"""1""","""origen (0,0)""",1927675,3,1927672,0.0002,"""D-005"""
"""2""","""origen fuera de Bolivia""",1927672,9,1927663,0.0005,"""D-005"""
"""3""","""copia de duplicado exacto""",1927663,60,1927603,0.0031,"""D-013"""
"""4""","""origen fuera del área de estud…",1927603,3568,1924035,0.1851,"""D-018"""
"""5""","""distancia > 30 km""",1924035,4962,1919073,0.2574,"""D-015"""
"""6""","""usuario anómalo""",1919073,233,1918840,0.0121,"""D-008"""


## T4 · Tabla minable por celda H3

La tabla parte de **todas** las celdas Kontur 2023 cuyo centroide cae en el área y se une con las celdas que tienen
consultas válidas: las celdas pobladas sin consultas quedan con `query_count = 0` (son las que el proyecto busca
identificar). Las celdas con consultas pero sin registro Kontur quedan con `population = 0`: no tienen exposición,
no entran al ajuste y se describen aparte como *polos de actividad*.

In [9]:
kontur_2023 = prep.load_kontur(KONTUR_2023_GPKG_GZ)
kontur_2022 = prep.load_kontur(KONTUR_2022_GPKG_GZ)
kontur_2023.write_parquet(INTERIM / "kontur_2023_h3r8.parquet")
kontur_2022.write_parquet(INTERIM / "kontur_2022_h3r8.parquet")

tabla = prep.build_cell_table(consultas_limpias, kontur_2023, AREA_ESTUDIO)
assert tabla["query_count"].sum() == N_LIMPIAS, "Suma de control T4 falla"
print(f"Celdas en la tabla: {tabla.height:,}")
print(f"  con consultas:            {(tabla['query_count'] > 0).sum():,}")
print(f"  con población > 0:        {(tabla['population'] > 0).sum():,}")
print(f"  pobladas sin consultas:   {((tabla['population'] > 0) & (tabla['query_count'] == 0)).sum():,}")
print(f"  población 0 y consultas:  {((tabla['population'] == 0) & (tabla['query_count'] > 0)).sum():,}")

Celdas en la tabla: 1,277
  con consultas:            885
  con población > 0:        1,240
  pobladas sin consultas:   392
  población 0 y consultas:  37


## T5 · Variables (D-105) y umbral de población (D-108)

| Grupo | Variables | Uso |
|---|---|---|
| Objetivo | `query_count` (alternativo: `user_count`) | y |
| Exposición | `population` | offset `log(population)` |
| Predictores territoriales | `dist_plaza_km`, `pop_ring1`, `pop_ring2` | modelo |
| **Contraste (nunca predictores)** | `dist_stop_m`, `gtfs_covered`, `route_count_500m` | solo Fase 5 (E5) |
| Validación | `block_id` (H3 res 6), `distance_ring` | pliegues y estratos |

`pop_ring1`/`pop_ring2` suman la población Kontur de la 1.ª y 2.ª corona (`h3.grid_ring`, sin la celda central),
usando todas las celdas Kontur de Bolivia para no subestimar el borde del área. Ninguna variable se deriva de las
consultas de otras celdas (regla dura 5).

In [10]:
tabla = prep.add_territorial_features(tabla, kontur_2023, AREA_ESTUDIO)
tabla = prep.add_gtfs_contrast(tabla, GTFS_DIR, UMBRAL_COBERTURA_M)
tabla = tabla.with_columns((pl.col("population") >= POP_MIN).alias("in_model"))

# Anillos de distancia (cuartiles de la distancia media a la Plaza por bloque, solo bloques con celdas del modelo)
bloques = (tabla.filter(pl.col("in_model")).group_by("block_id")
           .agg(pl.col("dist_plaza_km").mean().alias("block_dist_km"), pl.len().alias("n_cells"))
           .sort("block_id"))
cortes = np.quantile(bloques["block_dist_km"].to_numpy(), [0.25, 0.5, 0.75])
bloques = bloques.with_columns(
    pl.col("block_dist_km").cut(list(cortes), labels=["A1", "A2", "A3", "A4"]).cast(pl.Utf8).alias("distance_ring"))
tabla = tabla.join(bloques.select("block_id", "distance_ring"), on="block_id", how="left").with_columns(
    pl.col("distance_ring").fill_null("sin_modelo"))
print("Cortes de anillo (km):", np.round(cortes, 2))

Cortes de anillo (km): [10.7  16.04 20.5 ]


**Umbral de población mínima (D-108).** Con menos de 10 habitantes la tasa `query_count / population` es muy
inestable (una consulta equivale a > 0,1 consultas por habitante). Se marcan con `in_model = False` y se describen
aparte; la sensibilidad a `≥ 50` se evalúa en la Fase 5 (E9).

In [11]:
fuera_modelo = tabla.filter(~pl.col("in_model"))
polos = tabla.filter((pl.col("population") == 0) & (pl.col("query_count") > 0))
polos.select("h3_cell", "lat", "lon", "query_count", "user_count", "dist_plaza_km", "dist_stop_m") \
     .sort("query_count", descending=True).write_csv(REPORTE / "polos_actividad.csv")
umbral_pob = pl.DataFrame([
    {"grupo": "in_model (population >= 10)", "celdas": int(tabla["in_model"].sum()),
     "consultas": int(tabla.filter(pl.col("in_model"))["query_count"].sum()),
     "poblacion": int(tabla.filter(pl.col("in_model"))["population"].sum())},
    {"grupo": "1 <= population < 10", "celdas": fuera_modelo.filter(pl.col("population") > 0).height,
     "consultas": int(fuera_modelo.filter(pl.col("population") > 0)["query_count"].sum()),
     "poblacion": int(fuera_modelo["population"].sum())},
    {"grupo": "population = 0 con consultas (polos)", "celdas": polos.height,
     "consultas": int(polos["query_count"].sum()), "poblacion": 0},
])
umbral_pob.write_csv(REPORTE / "umbral_poblacion.csv")
umbral_pob

grupo,celdas,consultas,poblacion
str,i64,i64,i64
"""in_model (population >= 10)""",1087,1918628,1166068
"""1 <= population < 10""",153,121,430
"""population = 0 con consultas (…",37,91,0


### T5.5 · Persistir la tabla minable (una sola escritura + round-trip)

In [12]:
ORDEN = ["h3_cell", "query_count", "user_count", "population", "pop_ring1", "pop_ring2", "dist_plaza_km",
         "dist_stop_m", "gtfs_covered", "route_count_500m", "block_id", "distance_ring", "in_model", "edge_cell",
         "lat", "lon", "x_utm", "y_utm"]
tabla_final = tabla.select(ORDEN).sort("h3_cell")
tabla_final.write_parquet(PROCESSED / "tabla_minable.parquet", compression="snappy", use_pyarrow=True)

tabla_leida = pl.read_parquet(PROCESSED / "tabla_minable.parquet")
assert tabla_leida.shape == tabla_final.shape, "Pérdida de filas/columnas al escribir"
assert set(tabla_leida.columns) == set(tabla_final.columns), "Esquema distinto"
assert tabla_leida.equals(tabla_final), "Contenido distinto tras el round-trip"
(PROCESSED / "tabla_minable_dtypes.json").write_text(
    json.dumps({c: str(t) for c, t in tabla_final.schema.items()}, indent=2), encoding="utf-8")
print(f"tabla_minable.parquet OK: {tabla_leida.shape}")

tabla_minable.parquet OK: (1277, 18)


## T6 · Reserva de prueba: 20 % de bloques H3 res 6 por anillo (D-106)

Se sortea el 20 % de los bloques de cada anillo de distancia con semilla 42. El sorteo usa solo `block_id` y la
distancia a la Plaza (exógena): ninguna consulta interviene. Si el bloque del centro cae en prueba, se deja ahí.
**`test_blocks.csv` se versiona en Git antes de ajustar cualquier modelo.**

In [13]:
rng = np.random.default_rng(SEMILLA)
seleccion = []
for anillo in ["A1", "A2", "A3", "A4"]:
    ids = bloques.filter(pl.col("distance_ring") == anillo)["block_id"].sort().to_list()
    n = max(1, round(FRACCION_PRUEBA * len(ids)))
    seleccion += [(str(b), anillo) for b in rng.choice(ids, size=n, replace=False)]
test_blocks = (pl.DataFrame(seleccion, schema=["block_id", "distance_ring"], orient="row")
               .join(bloques.select("block_id", "block_dist_km", "n_cells"), on="block_id").sort("block_id"))
test_blocks.write_csv(PROCESSED / "test_blocks.csv")
test_blocks.write_csv(REPORTE / "test_blocks.csv")  # copia versionada (data/ está fuera de Git)

modelo = tabla_final.filter(pl.col("in_model")).with_columns(
    pl.col("block_id").is_in(test_blocks["block_id"].implode()).alias("in_test"))
particion = (modelo.group_by("in_test").agg(
    pl.col("block_id").n_unique().alias("bloques"), pl.len().alias("celdas"),
    pl.col("query_count").sum().alias("consultas"), pl.col("population").sum().alias("poblacion"))
    .with_columns(pl.when(pl.col("in_test")).then(pl.lit("prueba")).otherwise(pl.lit("entrenamiento")).alias("conjunto"))
    .with_columns((pl.col("consultas") / pl.col("consultas").sum() * 100).round(2).alias("pct_consultas"),
                  (pl.col("celdas") / pl.col("celdas").sum() * 100).round(2).alias("pct_celdas"))
    .sort("conjunto").drop("in_test"))
particion.write_csv(REPORTE / "particion_resumen.csv")
bloque_centro = h3.cell_to_parent(h3.latlng_to_cell(PLAZA_14_SEPTIEMBRE["lat"], PLAZA_14_SEPTIEMBRE["lon"], 8), 6)
print(f"Bloques de prueba: {test_blocks.height} de {bloques.height}")
print(f"Bloque de la Plaza ({bloque_centro}) en prueba: {bloque_centro in test_blocks['block_id'].to_list()}")
particion

Bloques de prueba: 8 de 43
Bloque de la Plaza (868b2c8a7ffffff) en prueba: False


bloques,celdas,consultas,poblacion,conjunto,pct_consultas,pct_celdas
u32,u32,i64,i64,str,f64,f64
35,840,1839109,983461,"""entrenamiento""",95.86,77.28
8,247,79519,182607,"""prueba""",4.14,22.72


## T7 · Auditoría de *leakage* L1–L8 (D-107)

In [14]:
predictores, contraste = prep.PREDICTORS, prep.CONTRAST
bloques_train = set(modelo.filter(~pl.col("in_test"))["block_id"])
bloques_test = set(modelo.filter(pl.col("in_test"))["block_id"])
celdas_test = set(modelo.filter(pl.col("in_test"))["h3_cell"])
celdas_train = set(modelo.filter(~pl.col("in_test"))["h3_cell"])
test_con_vecino_train = sum(any(n in celdas_train for n in h3.grid_ring(c, 1)) for c in celdas_test)
import subprocess
modelado_existe = (PROJECT_ROOT / "reports" / "03_modelado" / "cv_espacial.csv").exists()
_git = subprocess.run(["git", "show", "HEAD:reports/02_preparacion/test_blocks.csv"], cwd=PROJECT_ROOT,
                      capture_output=True, text=True)
test_blocks_versionado_igual = _git.returncode == 0 and _git.stdout == (REPORTE / "test_blocks.csv").read_text()

auditoria = [
    ("L1", "Objetivo dentro de los predictores", not set(predictores) & set(prep.TARGETS),
     f"predictores = {predictores}; objetivos = {prep.TARGETS}"),
    ("L2", "Predictores derivados de consultas de celdas vecinas", set(predictores) == {"dist_plaza_km", "pop_ring1", "pop_ring2"},
     "pop_ring* salen de Kontur; dist_plaza_km de un punto exógeno. B1 usa vecinos solo dentro del pliegue (Fase 4)."),
    ("L3", "Área definida con conteos de consultas", True,
     "Área = envolvente de orígenes de la componente de celdas ocupadas (ubicaciones); los conteos no intervienen (T1)."),
    ("L4", "Escalado o transformación ajustada antes de partir", True,
     "La tabla guarda variables crudas; log1p es fila a fila (sin parámetros); cualquier escalado ocurre dentro del pliegue (R3)."),
    ("L5", "Conjunto de prueba tocado antes de modelar", (not modelado_existe) or test_blocks_versionado_igual,
     f"test_blocks.csv se genera antes de cualquier modelo (modelado previo: {modelado_existe}); "
     f"idéntico a la versión en Git: {test_blocks_versionado_igual}."),
    ("L6", "Variables GTFS como predictores", not set(predictores) & set(contraste),
     f"contraste = {contraste} (D-017)."),
    ("L7", "Bloques compartidos entre entrenamiento y prueba", not bloques_train & bloques_test,
     f"{len(bloques_train)} bloques de entrenamiento, {len(bloques_test)} de prueba, 0 compartidos. "
     f"Celdas de prueba con vecina de entrenamiento (k=1, borde de bloque): {test_con_vecino_train} de {len(celdas_test)}."),
    ("L8", "Imputación con información global", tabla_final.null_count().sum_horizontal()[0] == 0,
     "Sin imputación: query_count/user_count = 0 y population = 0 son ausencias estructurales; 0 nulos en la tabla."),
]
auditoria_leakage = pl.DataFrame([{"id": i, "riesgo": r, "resultado": "OK" if ok else "FALLA", "evidencia": e}
                                  for i, r, ok, e in auditoria])
auditoria_leakage.write_csv(REPORTE / "auditoria_leakage.csv")
assert (auditoria_leakage["resultado"] == "OK").all(), "Auditoría de leakage con fallas"
auditoria_leakage

id,riesgo,resultado,evidencia
str,str,str,str
"""L1""","""Objetivo dentro de los predict…","""OK""","""predictores = ['dist_plaza_km'…"
"""L2""","""Predictores derivados de consu…","""OK""","""pop_ring* salen de Kontur; dis…"
"""L3""","""Área definida con conteos de c…","""OK""","""Área = envolvente de orígenes …"
"""L4""","""Escalado o transformación ajus…","""OK""","""La tabla guarda variables crud…"
"""L5""","""Conjunto de prueba tocado ante…","""OK""","""test_blocks.csv se genera ante…"
"""L6""","""Variables GTFS como predictore…","""OK""","""contraste = ['dist_stop_m', 'g…"
"""L7""","""Bloques compartidos entre entr…","""OK""","""35 bloques de entrenamiento, 8…"
"""L8""","""Imputación con información glo…","""OK""","""Sin imputación: query_count/us…"


## T8 · Verificaciones

### T8.1 · Suma de control, unicidad y verificaciones previas al modelado

In [15]:
v = [
    ("suma query_count = consultas limpias", int(tabla_final["query_count"].sum()) == N_LIMPIAS,
     f"{int(tabla_final['query_count'].sum()):,} vs {N_LIMPIAS:,}"),
    ("h3_cell sin duplicados", tabla_final["h3_cell"].is_duplicated().sum() == 0, f"{tabla_final.height:,} celdas"),
    ("sin solapamiento de celdas train/test", not celdas_train & celdas_test, "0 celdas compartidas"),
    ("population >= 10 en modelo", bool((modelo["population"] >= POP_MIN).all()), f"mín = {modelo['population'].min()}"),
    ("query_count >= 0", bool((tabla_final["query_count"] >= 0).all()), f"mín = {tabla_final['query_count'].min()}"),
    ("dist_plaza_km > 0", bool((tabla_final["dist_plaza_km"] > 0).all()), f"mín = {tabla_final['dist_plaza_km'].min():.3f}"),
    ("pop_ring1 >= 0 y pop_ring2 >= 0", bool((tabla_final["pop_ring1"] >= 0).all() & (tabla_final["pop_ring2"] >= 0).all()), "—"),
    ("pop_ring1 >= population (solo vale con grid_disk; aquí grid_ring)",
     None, f"{int((modelo['pop_ring1'] < modelo['population']).sum())} celdas del modelo con pop_ring1 < population (esperable: la corona excluye la celda central)"),
    ("sin nulos en ninguna columna", tabla_final.null_count().sum_horizontal()[0] == 0, "0 nulos"),
    ("block_id no nulo", tabla_final["block_id"].null_count() == 0, "—"),
]
verificaciones = pl.DataFrame([{"verificacion": n, "resultado": "INFO" if ok is None else ("OK" if ok else "FALLA"), "detalle": d}
                               for n, ok, d in v])
verificaciones.write_csv(REPORTE / "verificaciones_modelado.csv")
assert (verificaciones["resultado"] != "FALLA").all()
verificaciones

verificacion,resultado,detalle
str,str,str
"""suma query_count = consultas l…","""OK""","""1,918,840 vs 1,918,840"""
"""h3_cell sin duplicados""","""OK""","""1,277 celdas"""
"""sin solapamiento de celdas tra…","""OK""","""0 celdas compartidas"""
"""population >= 10 en modelo""","""OK""","""mín = 10"""
"""query_count >= 0""","""OK""","""mín = 0"""
"""dist_plaza_km > 0""","""OK""","""mín = 0.511"""
"""pop_ring1 >= 0 y pop_ring2 >= …","""OK""","""—"""
"""pop_ring1 >= population (solo …","""INFO""","""14 celdas del modelo con pop_r…"
"""sin nulos en ninguna columna""","""OK""","""0 nulos"""


### T8.2 · Correlograma de Moran (k-ring 1–6)

I de Moran de `log1p(tasa por 1.000 hab.)` sobre las celdas del modelo, con vecinos a distancia H3 **exacta** k
(`grid_ring(c, k)`), pesos estandarizados por fila y 999 permutaciones. Sirve para comparar el alcance de la
autocorrelación con el tamaño del bloque res 6 (arista ≈ 3,7 km; ≈ 7 km de ancho).

In [16]:
DIST_ENTRE_CENTROS_KM = 2 * h3.average_hexagon_edge_length(8, unit="km") * np.sqrt(3) / 2
tasa_log = np.log1p(modelo["query_count"].to_numpy() / modelo["population"].to_numpy() * 1000)
celdas_modelo = modelo["h3_cell"].to_list()
indice = {c: i for i, c in enumerate(celdas_modelo)}
correlograma = []
for k in range(1, 7):
    vecinos = {i: [indice[n] for n in h3.grid_ring(c, k) if n in indice] for i, c in enumerate(celdas_modelo)}
    con_vecinos = [i for i, vs in vecinos.items() if vs]
    remap = {i: j for j, i in enumerate(con_vecinos)}
    w = libpysal.weights.W({remap[i]: [remap[n] for n in vecinos[i] if n in remap] for i in con_vecinos},
                           silence_warnings=True)
    w.transform = "r"
    mi = Moran(tasa_log[con_vecinos], w, permutations=999)
    correlograma.append({"k": k, "distancia_aprox_km": round(k * DIST_ENTRE_CENTROS_KM, 2), "moran_I": round(mi.I, 4),
                         "EI": round(mi.EI, 4), "z_sim": round(mi.z_sim, 2), "p_sim": mi.p_sim, "celdas": len(con_vecinos)})
moran_corr = pl.DataFrame(correlograma)
moran_corr.write_csv(REPORTE / "moran_correlograma.csv")

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(moran_corr["distancia_aprox_km"], moran_corr["moran_I"], "o-", color="#1976D2")
ax.axhline(0, color="grey", lw=0.8)
ax.axvline(2 * h3.average_hexagon_edge_length(6, unit="km"), color="#E65100", ls="--", lw=1,
           label="ancho aprox. bloque res 6")
ax.set_xlabel("Distancia entre centroides (km, anillo H3 k)"); ax.set_ylabel("I de Moran")
ax.set_title(f"Correlograma de log1p(consultas/1.000 hab.) — N={len(celdas_modelo):,} celdas")
ax.legend(); fig.tight_layout()
guardar_figura(fig, "moran_correlograma", SUB); plt.close(fig)
moran_corr

k,distancia_aprox_km,moran_I,EI,z_sim,p_sim,celdas
i64,f64,f64,f64,f64,f64,i64
1,0.92,0.7224,-0.0009,37.3,0.001,1077
2,1.84,0.6271,-0.0009,41.44,0.001,1080
3,2.76,0.5495,-0.0009,45.08,0.001,1081
4,3.68,0.4678,-0.0009,42.98,0.001,1085
5,4.6,0.4054,-0.0009,39.19,0.001,1087
6,5.52,0.3527,-0.0009,35.53,0.001,1087


### T8.3 · Concentración (Gini) y cobertura GTFS dentro del área

In [17]:
lorenz_y = np.cumsum(np.sort(modelo["query_count"].to_numpy())) / modelo["query_count"].sum()
top10 = np.sort(tabla_final["query_count"].to_numpy())[::-1][:10].sum() / N_LIMPIAS
concentracion = pl.DataFrame([
    {"universo": "celdas del modelo (incluye ceros)", "celdas": modelo.height,
     "gini_query_count": round(prep.gini(modelo["query_count"].to_numpy()), 4),
     "pct_celdas_cero": round((modelo["query_count"] == 0).mean() * 100, 2)},
    {"universo": "todas las celdas de la tabla", "celdas": tabla_final.height,
     "gini_query_count": round(prep.gini(tabla_final["query_count"].to_numpy()), 4),
     "pct_celdas_cero": round((tabla_final["query_count"] == 0).mean() * 100, 2)},
    {"universo": "solo celdas con consultas", "celdas": int((tabla_final["query_count"] > 0).sum()),
     "gini_query_count": round(prep.gini(tabla_final.filter(pl.col("query_count") > 0)["query_count"].to_numpy()), 4),
     "pct_celdas_cero": 0.0},
]).with_columns(pl.lit(round(top10 * 100, 2)).alias("pct_consultas_top10_celdas"))
concentracion.write_csv(REPORTE / "concentracion.csv")
concentracion

universo,celdas,gini_query_count,pct_celdas_cero,pct_consultas_top10_celdas
str,i64,f64,f64,f64
"""celdas del modelo (incluye cer…",1087,0.9339,24.75,42.08
"""todas las celdas de la tabla""",1277,0.9437,30.7,42.08
"""solo celdas con consultas""",885,0.9187,0.0,42.08


In [18]:
def resumen_cobertura(df: pl.DataFrame, universo: str) -> pl.DataFrame:
    return (df.group_by("gtfs_covered").agg(
        pl.len().alias("celdas"), pl.col("query_count").sum().alias("consultas"),
        pl.col("population").sum().alias("poblacion"), (pl.col("query_count") == 0).sum().alias("celdas_sin_consultas"))
        .with_columns(pl.lit(universo).alias("universo"),
                      (pl.col("celdas") / pl.col("celdas").sum() * 100).round(2).alias("pct_celdas"),
                      (pl.col("consultas") / pl.col("consultas").sum() * 100).round(2).alias("pct_consultas"),
                      (pl.col("consultas") / pl.col("poblacion") * 1000).round(2).alias("consultas_por_1000_hab"))
        .sort("gtfs_covered"))

resumen_cob = pl.concat([resumen_cobertura(tabla_final, "todas las celdas"), resumen_cobertura(modelo, "celdas del modelo")])
resumen_cob.select("universo", pl.exclude("universo")).write_csv(REPORTE / "resumen_cobertura_gtfs.csv")
resumen_cob

gtfs_covered,celdas,consultas,poblacion,celdas_sin_consultas,universo,pct_celdas,pct_consultas,consultas_por_1000_hab
i8,u32,i64,i64,u32,str,f64,f64,f64
0,678,13367,154799,367,"""todas las celdas""",53.09,0.7,86.35
1,599,1905473,1011699,25,"""todas las celdas""",46.91,99.3,1883.44
0,494,13176,154378,245,"""celdas del modelo""",45.45,0.69,85.35
1,593,1905452,1011690,24,"""celdas del modelo""",54.55,99.31,1883.43


### T8.4 · Resúmenes por municipio y por anillo

In [19]:
municipio_celda = (consultas_limpias
                   .with_columns(pl.col("origin_municipio").map_elements(prep.fix_mojibake, return_dtype=pl.Utf8))
                   .group_by("h3_origin", "origin_municipio").len()
                   .sort(["h3_origin", "len"], descending=[False, True])
                   .group_by("h3_origin").first().rename({"h3_origin": "h3_cell", "origin_municipio": "municipio_modal"})
                   .select("h3_cell", "municipio_modal"))
por_municipio = (tabla_final.join(municipio_celda, on="h3_cell", how="left")
                 .with_columns(pl.col("municipio_modal").fill_null("(sin consultas)"))
                 .group_by("municipio_modal").agg(
                     pl.len().alias("celdas"), pl.col("query_count").sum().alias("consultas"),
                     pl.col("population").sum().alias("poblacion"), pl.col("in_model").sum().alias("celdas_modelo"))
                 .with_columns((pl.col("consultas") / pl.col("consultas").sum() * 100).round(2).alias("pct_consultas"))
                 .sort("consultas", descending=True))
por_municipio.write_csv(REPORTE / "resumen_por_municipio.csv")
por_municipio

municipio_modal,celdas,consultas,poblacion,celdas_modelo,pct_consultas
str,u32,i64,i64,u32,f64
"""Cochabamba""",302,1658662,618491,280,86.44
"""Sacaba""",180,89874,169364,166,4.68
"""Quillacollo""",118,65012,142843,110,3.39
"""Colcapirhua""",35,53066,47652,35,2.77
"""Tiquipaya""",45,46647,60789,42,2.43
…,…,…,…,…,…
"""Arbieto""",39,508,6728,34,0.03
"""Tolata""",28,286,4706,26,0.01
"""Villa Santivañez""",14,88,167,4,0.0


In [20]:
por_anillo = (modelo.join(bloques.select("block_id", "block_dist_km"), on="block_id")
              .group_by("distance_ring").agg(
                  pl.col("block_id").n_unique().alias("bloques"), pl.len().alias("celdas"),
                  pl.col("query_count").sum().alias("consultas"), pl.col("population").sum().alias("poblacion"),
                  pl.col("dist_plaza_km").min().round(2).alias("dist_min_km"), pl.col("dist_plaza_km").max().round(2).alias("dist_max_km"),
                  pl.col("in_test").sum().alias("celdas_prueba"))
              .with_columns((pl.col("consultas") / pl.col("poblacion") * 1000).round(2).alias("consultas_por_1000_hab"),
                            (pl.col("consultas") / pl.col("consultas").sum() * 100).round(2).alias("pct_consultas"))
              .sort("distance_ring"))
por_anillo.write_csv(REPORTE / "resumen_por_anillo.csv")
por_anillo

distance_ring,bloques,celdas,consultas,poblacion,dist_min_km,dist_max_km,celdas_prueba,consultas_por_1000_hab,pct_consultas
str,u32,u32,i64,i64,f64,f64,u32,f64,f64
"""A1""",11,416,1848877,830447,0.51,13.83,84,2226.36,96.36
"""A2""",11,318,66814,232110,8.95,18.99,78,287.85,3.48
"""A3""",10,188,2178,59968,14.33,22.41,38,36.32,0.11
"""A4""",11,165,759,43543,18.45,27.86,47,17.43,0.04


### T8.5 · Diccionario de datos (R5 → Anexo A)

In [21]:
descripciones = {
    "h3_cell": ("ID celda H3 res 8", "identificador"),
    "query_count": ("Consultas válidas con origen en la celda (sep-2022 a jun-2024)", "objetivo"),
    "user_count": ("Usuarios (instalaciones) distintos con origen en la celda", "objetivo alternativo"),
    "population": ("Población Kontur 2023 de la celda", "exposición (offset)"),
    "pop_ring1": ("Suma de población Kontur en la 1.ª corona H3 (grid_ring k=1)", "predictor (log1p)"),
    "pop_ring2": ("Suma de población Kontur en la 2.ª corona H3 (grid_ring k=2)", "predictor (log1p)"),
    "dist_plaza_km": ("Distancia haversine del centroide a la Plaza 14 de Septiembre", "predictor"),
    "dist_stop_m": ("Distancia (UTM 19S) del centroide a la parada GTFS más cercana", "contraste — NO predictor"),
    "gtfs_covered": ("1 si dist_stop_m <= 500", "contraste — NO predictor"),
    "route_count_500m": ("Líneas GTFS (route_id) distintas con parada a <= 500 m", "contraste — NO predictor"),
    "block_id": ("Bloque H3 res 6 (padre de la celda)", "grupo de validación"),
    "distance_ring": ("Anillo A1–A4 por cuartil de distancia media del bloque a la Plaza", "estrato de validación"),
    "in_model": ("True si population >= 10", "filtro de modelado"),
    "edge_cell": ("True si el centroide está a <= 1 km del borde del área", "diagnóstico"),
    "lat": ("Latitud del centroide (EPSG:4326)", "geometría"),
    "lon": ("Longitud del centroide (EPSG:4326)", "geometría"),
    "x_utm": ("X del centroide (EPSG:32719)", "geometría"),
    "y_utm": ("Y del centroide (EPSG:32719)", "geometría"),
}
filas_dic = []
for col, dtype in tabla_final.schema.items():
    s = tabla_final[col]
    numerica = dtype.is_numeric() or dtype == pl.Boolean
    sn = s.cast(pl.Float64) if numerica else None
    filas_dic.append({
        "columna": col, "tipo": str(dtype), "rol": descripciones[col][1], "nulos": s.null_count(),
        "pct_nulos": round(s.null_count() / s.len() * 100, 2),
        "min": round(sn.min(), 3) if numerica else None, "max": round(sn.max(), 3) if numerica else None,
        "pct_ceros": round((sn == 0).mean() * 100, 2) if numerica else None,
        "distintos": s.n_unique(), "descripcion": descripciones[col][0],
    })
diccionario = pl.DataFrame(filas_dic)
diccionario.write_csv(REPORTE / "diccionario_datos.csv")
diccionario

columna,tipo,rol,nulos,pct_nulos,min,max,pct_ceros,distintos,descripcion
str,str,str,i64,f64,f64,f64,f64,i64,str
"""h3_cell""","""String""","""identificador""",0,0.0,null,null,null,1277,"""ID celda H3 res 8"""
"""query_count""","""Int64""","""objetivo""",0,0.0,0.0,152201.0,30.7,398,"""Consultas válidas con origen e…"
"""user_count""","""Int64""","""objetivo alternativo""",0,0.0,0.0,34315.0,30.7,327,"""Usuarios (instalaciones) disti…"
"""population""","""Int64""","""exposición (offset)""",0,0.0,0.0,5240.0,2.9,773,"""Población Kontur 2023 de la ce…"
"""pop_ring1""","""Float64""","""predictor (log1p)""",0,0.0,1.0,28988.0,0.0,1116,"""Suma de población Kontur en la…"
…,…,…,…,…,…,…,…,…,…
"""edge_cell""","""Boolean""","""diagnóstico""",0,0.0,0.0,1.0,90.21,2,"""True si el centroide está a <=…"
"""lat""","""Float64""","""geometría""",0,0.0,-17.575,-17.282,0.0,1277,"""Latitud del centroide (EPSG:43…"
"""lon""","""Float64""","""geometría""",0,0.0,-66.397,-65.903,0.0,1277,"""Longitud del centroide (EPSG:4…"


### T8.6 · Figuras F1–F5

Mapas en EPSG:4326; cálculos en UTM 19S. Colormap secuencial para conteos. Las figuras se guardan en PNG
(`reports/02_preparacion/figuras/`) y no se muestran.

In [22]:
gdf_celdas = gpd.GeoDataFrame(tabla_final.to_pandas(), geometry=prep.cell_polygons(tabla_final["h3_cell"].to_list()),
                              crs="EPSG:4326")
area_gs = gpd.GeoSeries([AREA_ESTUDIO], crs="EPSG:4326")
gtfs_gs = gpd.GeoSeries([AREA_GTFS], crs="EPSG:4326")
minx, miny, maxx, maxy = AREA_ESTUDIO.union(AREA_GTFS).bounds
MARCO = dict(xlim=(minx - 0.03, maxx + 0.03), ylim=(miny - 0.03, maxy + 0.03))

# F1 · Área de estudio y consultas dentro/fuera
base_f1 = consultas.filter(coords_ok & ~pl.col("flag_duplicate_copy"))
dentro_f1 = base_f1.filter(pl.col("in_area")).sample(200_000, seed=SEMILLA)
fuera_f1 = base_f1.filter(~pl.col("in_area"))
n_fuera_marco = fuera_f1.filter(~(pl.col("lon_origin").is_between(*MARCO["xlim"]) & pl.col("lat_origin").is_between(*MARCO["ylim"]))).height
fig, ax = plt.subplots(figsize=(11, 9))
area_gs.plot(ax=ax, color="#E3F2FD", edgecolor="#1976D2", linewidth=1.5)
gtfs_gs.boundary.plot(ax=ax, color="#616161", linewidth=1, linestyle="--")
ax.scatter(dentro_f1["lon_origin"], dentro_f1["lat_origin"], s=1, alpha=0.05, color="#2196F3")
ax.scatter(fuera_f1["lon_origin"], fuera_f1["lat_origin"], s=3, alpha=0.4, color="#FF9800")
ax.set(**MARCO, xlabel="Longitud", ylabel="Latitud")
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
ax.legend(handles=[Patch(facecolor="#E3F2FD", edgecolor="#1976D2", label=f"Área de estudio Etapa 2 ({km2_etapa2:,.0f} km²)"),
                   Line2D([], [], color="#616161", ls="--", label=f"Área Etapa 1, hull GTFS ({km2_etapa1:,.0f} km²)"),
                   Line2D([], [], marker="o", ls="", color="#2196F3", label=f"Dentro: {n_en_area:,} (muestra 200k)"),
                   Line2D([], [], marker="o", ls="", color="#FF9800", label=f"Fuera: {fuera_f1.height:,} ({n_fuera_marco} fuera del marco)")],
          loc="upper right", fontsize=9)
ax.set_title(f"Área de estudio (hull orígenes válidos + 1 km) — N={base_f1.height:,} consultas con coordenadas válidas")
fig.tight_layout(); guardar_figura(fig, "area_estudio_y_consultas", SUB); plt.close(fig)

# F2 · Histograma de distancia con línea en 30 km
dist_f2 = base_f1.filter(pl.col("in_area") & (pl.col("distancia") > 0))["distancia"].to_numpy() / 1000
fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(dist_f2, bins=np.logspace(np.log10(max(dist_f2.min(), 0.01)), np.log10(dist_f2.max()), 80), color="#90CAF9", edgecolor="white")
ax.set_xscale("log"); ax.set_yscale("log")
ax.axvline(DIST_MAX_M / 1000, color="#D32F2F", lw=2, label=f"Umbral 30 km: {(dist_f2 > 30).sum():,} excluidas ({(dist_f2 > 30).mean()*100:.2f} %)")
for p, ls in [(50, ":"), (99, "--")]:
    ax.axvline(np.percentile(dist_f2, p), color="#424242", ls=ls, lw=1, label=f"P{p} = {np.percentile(dist_f2, p):.1f} km")
ax.set_xlabel("distancia origen–destino (km, escala log)"); ax.set_ylabel("consultas (escala log)")
ax.set_title(f"Distribución de distancia — consultas con origen en el área, N={dist_f2.size:,}")
ax.legend(); fig.tight_layout(); guardar_figura(fig, "distancia_histograma", SUB); plt.close(fig)

# F3 · gtfs_covered
fig, ax = plt.subplots(figsize=(11, 9))
gdf_celdas.plot(ax=ax, column=gdf_celdas["gtfs_covered"].map({1: "Cubierta (≤ 500 m)", 0: "No cubierta"}),
                categorical=True, cmap="Paired", legend=True, edgecolor="white", linewidth=0.1,
                legend_kwds={"loc": "upper right"})
area_gs.boundary.plot(ax=ax, color="#1976D2", linewidth=1.2)
n_cub = int(gdf_celdas["gtfs_covered"].sum())
ax.set_title(f"gtfs_covered (contraste, no predictor) — {n_cub:,} de {len(gdf_celdas):,} celdas cubiertas ({n_cub/len(gdf_celdas)*100:.1f} %)")
ax.set(**MARCO, xlabel="Longitud", ylabel="Latitud")
fig.tight_layout(); guardar_figura(fig, "gtfs_cobertura_mapa", SUB); plt.close(fig)

# F4 · query_count en escala log
fig, ax = plt.subplots(figsize=(11, 9))
gdf_celdas.assign(log_q=np.log10(gdf_celdas["query_count"] + 1)).plot(
    ax=ax, column="log_q", cmap="viridis", legend=True, edgecolor="none",
    legend_kwds={"label": "log10(query_count + 1)", "shrink": 0.7})
area_gs.boundary.plot(ax=ax, color="#1976D2", linewidth=1.2)
ax.plot(PLAZA_14_SEPTIEMBRE["lon"], PLAZA_14_SEPTIEMBRE["lat"], "r*", ms=12, label="Plaza 14 de Septiembre")
ax.set_title(f"query_count por celda (log) — {len(gdf_celdas):,} celdas, {N_LIMPIAS:,} consultas válidas")
ax.set(**MARCO, xlabel="Longitud", ylabel="Latitud"); ax.legend(loc="upper right")
fig.tight_layout(); guardar_figura(fig, "query_count_mapa", SUB); plt.close(fig)

# F5 · Curva de Lorenz
fig, ax = plt.subplots(figsize=(7, 6))
x_l = np.arange(1, modelo.height + 1) / modelo.height
ax.plot(np.r_[0, x_l] * 100, np.r_[0, lorenz_y] * 100, color="#1976D2", lw=2, label="Curva de Lorenz")
ax.plot([0, 100], [0, 100], "k--", lw=1, alpha=0.5, label="Equidistribución")
ax.set_xlabel("% de celdas (orden creciente de query_count)"); ax.set_ylabel("% de consultas acumuladas")
ax.set_title(f"Lorenz de query_count — N={modelo.height:,} celdas del modelo\n"
             f"Gini = {prep.gini(modelo['query_count'].to_numpy()):.3f}; {(modelo['query_count'] == 0).mean()*100:.1f} % celdas en cero")
ax.legend(); fig.tight_layout(); guardar_figura(fig, "lorenz_query_count", SUB); plt.close(fig)

# Extra · bloques de prueba
fig, ax = plt.subplots(figsize=(11, 9))
g_mod = gdf_celdas[gdf_celdas["in_model"]].copy()
g_mod["conjunto"] = np.where(g_mod["block_id"].isin(test_blocks["block_id"].to_list()), "prueba", "entrenamiento")
g_mod.plot(ax=ax, column="conjunto", categorical=True, cmap="Set2", legend=True, edgecolor="none")
area_gs.boundary.plot(ax=ax, color="#1976D2", linewidth=1.2)
ax.plot(PLAZA_14_SEPTIEMBRE["lon"], PLAZA_14_SEPTIEMBRE["lat"], "r*", ms=12)
ax.set_title(f"Reserva de prueba: {test_blocks.height} de {bloques.height} bloques H3 res 6 (semilla {SEMILLA})")
ax.set(**MARCO, xlabel="Longitud", ylabel="Latitud")
fig.tight_layout(); guardar_figura(fig, "particion_bloques", SUB); plt.close(fig)
print(sorted(p.name for p in (REPORTE / "figuras").glob("*.png")))

['area_estudio_y_consultas.png', 'distancia_histograma.png', 'gtfs_cobertura_mapa.png', 'lorenz_query_count.png', 'moran_correlograma.png', 'particion_bloques.png', 'query_count_mapa.png']


## T9 · README de la fase

Se regenera desde este notebook para que cada cifra salga de un CSV de `reports/02_preparacion/`.

In [23]:
def md_tabla(df: pl.DataFrame) -> str:
    cols = df.columns
    filas = ["| " + " | ".join(cols) + " |", "|" + "---|" * len(cols)]
    for r in df.iter_rows():
        filas.append("| " + " | ".join("" if x is None else (f"{x:,}" if isinstance(x, int) and not isinstance(x, bool) else str(x)) for x in r) + " |")
    return "\n".join(filas)

cmp = dict(zip(comparacion_area["medida"], comparacion_area["valor"]))
readme = f"""# Etapa 2 · Preparación de datos (7.3)

Generado por `notebooks/02_preparacion_datos.ipynb`. Decisiones: `DECISIONES_02_preparacion.md` (D-101 a D-110).

> **Cambio de área respecto a la Etapa 1.** El área de estudio ya no es la envolvente convexa del feed GTFS (D-009, superada por D-018). Es la envolvente convexa de los orígenes de consultas válidas (componente espacial principal) + 1 km. GTFS pasa a ser solo variable de contraste.

## Área de estudio

```
Área Etapa 1 (hull GTFS + 1 km):     {cmp['area_etapa1_hull_gtfs_km2']:,} km²
Área Etapa 2 (hull orígenes + 1 km): {cmp['area_etapa2_hull_origenes_km2']:,} km²
Diferencia:                          {cmp['diferencia_km2']:,} km²  ({cmp['pct_etapa1_conservado']} % del área de la Etapa 1 se conserva)
Consultas que caen dentro del área de la Etapa 2: {int(cmp['consultas_coord_ok_dentro_area_etapa2']):,} / {int(cmp['consultas_coord_ok']):,} ({cmp['pct_consultas_dentro_area_etapa2']} %)
```

Variantes evaluadas (`area_estudio_variantes.csv`):

{md_tabla(tabla_variantes)}

## Flujo de limpieza (`tabla_flujo_limpieza.csv`)

{md_tabla(tabla_flujo)}

Umbral de distancia: `sensibilidad_umbral_km.csv`.

{md_tabla(sens_umbral)}

## Tabla minable (`data/processed/tabla_minable.parquet`)

- {tabla_final.height:,} celdas H3 r8; {int((tabla_final['query_count'] > 0).sum()):,} con consultas; {int(((tabla_final['population'] > 0) & (tabla_final['query_count'] == 0)).sum()):,} pobladas sin consultas.
- Celdas del modelo (`population ≥ {POP_MIN}`): {modelo.height:,}; polos de actividad (población 0 con consultas): {polos.height:,} (`polos_actividad.csv`).

{md_tabla(umbral_pob)}

### Variables para el modelado (Etapa 3)

| Variable | Rol en el modelo | Tipo |
|---|---|---|
| `query_count` | Objetivo | count |
| `population` | Offset `log(population)` | exposure |
| `dist_plaza_km` | Predictor | continuous |
| `pop_ring1` | Predictor (log1p) | continuous |
| `pop_ring2` | Predictor (log1p) | continuous |
| `gtfs_covered` | **Contraste post-hoc, NO predictor** | binary |
| `dist_stop_m` | **Contraste post-hoc, NO predictor** | continuous |
| `route_count_500m` | **Contraste post-hoc, NO predictor** | count |
| `block_id` | Grupo de validación | categorical |

## Reserva de prueba (`test_blocks.csv`)

{md_tabla(particion)}

Bloque de la Plaza (`{bloque_centro}`) en prueba: **{bloque_centro in test_blocks['block_id'].to_list()}**.

## Verificaciones

{md_tabla(verificaciones)}

### Auditoría de leakage (`auditoria_leakage.csv`)

{md_tabla(auditoria_leakage.select('id', 'riesgo', 'resultado'))}

### Correlograma de Moran (`moran_correlograma.csv`)

{md_tabla(moran_corr)}

### Concentración y cobertura

{md_tabla(concentracion)}

{md_tabla(resumen_cob.select('universo', 'gtfs_covered', 'celdas', 'pct_celdas', 'consultas', 'pct_consultas', 'consultas_por_1000_hab'))}

## Resúmenes

- `resumen_por_municipio.csv`, `resumen_por_anillo.csv`, `resumen_cobertura_gtfs.csv`, `diccionario_datos.csv` (Anexo A).

{md_tabla(por_anillo)}

## Figuras (`figuras/`)

| Figura | Archivo |
|---|---|
| F1 Área y consultas dentro/fuera | `area_estudio_y_consultas.png` |
| F2 Histograma de distancia (30 km) | `distancia_histograma.png` |
| F3 Mapa de `gtfs_covered` | `gtfs_cobertura_mapa.png` |
| F4 Mapa de `query_count` (log) | `query_count_mapa.png` |
| F5 Lorenz de `query_count` | `lorenz_query_count.png` |
| Extra: correlograma de Moran | `moran_correlograma.png` |
| Extra: bloques de prueba | `particion_bloques.png` |
"""
(REPORTE / "README.md").write_text(readme, encoding="utf-8")
print(readme[:1500])

# Etapa 2 · Preparación de datos (7.3)

Generado por `notebooks/02_preparacion_datos.ipynb`. Decisiones: `DECISIONES_02_preparacion.md` (D-101 a D-110).

> **Cambio de área respecto a la Etapa 1.** El área de estudio ya no es la envolvente convexa del feed GTFS (D-009, superada por D-018). Es la envolvente convexa de los orígenes de consultas válidas (componente espacial principal) + 1 km. GTFS pasa a ser solo variable de contraste.

## Área de estudio

```
Área Etapa 1 (hull GTFS + 1 km):     1,372.9 km²
Área Etapa 2 (hull orígenes + 1 km): 1,181.4 km²
Diferencia:                          -191.4 km²  (79.8 % del área de la Etapa 1 se conserva)
Consultas que caen dentro del área de la Etapa 2: 1,924,094 / 1,927,663 (99.81 %)
```

Variantes evaluadas (`area_estudio_variantes.csv`):

| variante | celdas_ocupadas | area_km2 | consultas_validas_dentro | pct_consultas_validas_dentro | adoptada |
|---|---|---|---|---|---|
| todas_las_celdas | 1,146 | 40967.3 | 1,920,618 | 100.0 | False |
| c